# Notebook 08 — retrospective Morning recall-window comparison

**Question:** do HR mean, SD, latest and slope from reported waking to the existing pre-EMA cutoff
predict Morning's three-item perceived-stress composite better than fixed windows on identical responses?

The [specification](../reports/morning_recall_specification.md) was frozen before these performance
calculations. Primary comparison: wake versus **60 minutes**; secondary: **5 and 120 minutes**.
No window search. Both wake-minute scenarios and the all-window common cohort are reported.
This follows inspection of Notebook 07 and is exploratory, **retrospective**, not an independent
confirmation or a deployable prediction model. The current-survey wake answer defines an interval;
it is never a predictor or assumed available before the response. All 32 final-test participants remain sealed.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from garmin_preprocessing.common import verify_bundle  # noqa: E402
from garmin_hr_activity.features import CLOCK_FLAGS, prepare  # noqa: E402
from garmin_morning_recall.data import wake_features  # noqa: E402

BUNDLE = ROOT / "private/morning_recall/runs/4c844f737b74ee62"
REPORT = ROOT / "reports/morning_recall/b1fd2e61d20cdbe4"
manifest = verify_bundle(BUNDLE)
verify_bundle(REPORT)
assert manifest["morning_emas_accounted"] == 934
assert manifest["retrospective_only"] and manifest["development_only"]
print({key: manifest[key] for key in ["morning_emas_accounted", "feature_rows", "quality_rows", "prediction_rows", "fitted_models"]})
print("Package versions:", manifest["inputs"]["versions"])

## Synthetic timeline: the same right edge, different left edges

Suppose actual start is 08:10 with minute precision, so cutoff is 08:09. A wake answer of 07:00
produces retrospective [07:00,08:09). The fixed 60-minute interval begins 07:09; 5-minute begins
08:04; 120-minute begins 06:09 (before the reported awakening). All right edges are excluded.
The later-minute wake sensitivity starts 07:01. With local-minus-UTC offset −240, add four hours
for UTC. No actual participant clock is displayed here. The one-minute sensitivity concerns
recorded granularity, not a confidence interval for recall error.

In [ ]:
display(Image(filename=str(REPORT / "synthetic_timeline.png")))
synthetic = pd.DataFrame({"n_epoch_s": [0., 60., 120., 180.], "n_value": [60., 70., 80., 200.],
    "n_logical_row": [0, 1, 2, 3], "q_conflicting_timestamp_value": False,
    **{name: False for name in CLOCK_FLAGS}})
hr_rules = json.loads((ROOT / "config/hr_activity.json").read_text())
groups, _ = prepare(synthetic, "hr", hr_rules)
values, quality, _ = wake_features(groups, 0., 180., hr_rules)
assert np.allclose(list(values.values()), [70., 10., 80., 10.])
print("Synthetic [0,180) features:", values)
print("The 180-second reading is excluded. The sample SD uses N−1.")
short_values, short_quality, _ = wake_features(groups, 120., 180., hr_rules)
print("Synthetic single-reading interval:", short_values)
print("A latest reading alone cannot enter the four-feature matched comparison.")

## Preserve all Morning responses and existing rules

All 934 Morning responses remain in each scenario's accounting, including three missing scores,
invalid/missing wake boundaries, absent recordings and insufficient feature support. The existing
composite and stored item coding are unchanged. There is no outcome imputation.

HR acceptance is unchanged: finite clocks/values, 30–240 bpm, clock flags, conflicting-timestamp
group rejection and same-timestamp/value collapse. Mean and sample SD require two distinct accepted
timestamps spanning one second; latest requires one; slope requires three spanning 60 seconds.
These are provisional support rules, not validated continuous wear. No interpolation, carry-forward,
extra dates or threshold relaxation. Quality and wake clock/duration stay outside the predictor list.

In [ ]:
print(json.loads((REPORT / "morning_accounting_totals.json").read_text()))
availability = pd.read_json(REPORT / "availability.json")
display(availability[["scenario", "interval", "responses", "boundary_unavailable", "no_recording",
                      "all_readings_rejected", "any_accepted_hr", "insufficient_feature_support", "complete_hr"]])
display(Image(filename=str(REPORT / "availability.png")))

## Matching protects the comparison

Each pairwise cohort requires a valid score/context/boundary and all four features in both intervals.
Within every original participant fold, **every comparator uses identical training and validation rows**.
Dummies/context do not borrow a larger training population. The common cohort additionally requires
all four intervals; it supplements rather than replaces the primary pairwise cohort.

The primary cohort retains 227 responses / 86 participants (24.3% of Morning responses); 5 minutes
and the common cohort retain 191 / 74; 120 minutes retains 233 / 87. Both wake sensitivities retain
the same comparison memberships. All folds are feasible. The inherited small-cohort flags do not
trigger, but these remain modest, selected cohorts. Participants can contribute included and excluded
responses; participant counts cannot be added. The common/5-minute cohorts are repeated views of
the same responses, not independent confirmations.

In [ ]:
cohorts = pd.read_json(REPORT / "cohorts.json")
folds = pd.read_json(REPORT / "fold_counts.json")
composition = pd.read_json(REPORT / "composition.json")
display(cohorts[["scenario", "cohort", "responses", "participants", "excluded_responses", "eligible_without_oof", "limited_interpretation"]])
nominal = "reported_minute_start"
display(folds.loc[folds.scenario.eq(nominal) & folds.cohort.eq("pair_fixed_60")])
display(composition.loc[composition.scenario.eq(nominal) & composition.cohort.eq("pair_fixed_60"),
    ["group", "responses", "participants", "valid_scores", "missing_scores", "mean", "median", "q25", "q75"]])

## Fixed models, held-out errors and conditional uncertainty

Models are matching training-mean/median dummies, context-only Ridge and context+HR Ridge for each
interval. Context is actual-start cyclic local time. The existing category encoder has only Morning,
so its constant column adds no contrast. All preprocessing fits on training folds only. Ridge alpha=1,
solver=svd is unchanged. No HR-only model, quantile regression, alpha search, quality predictor or
current-answer predictor. An illustrative score from items (4,2,3) is 4+(6−2)+(6−3)=11; its held-out
prediction comes from models trained on other participants with the same cohort membership.

MAE is primary; RMSE secondary. Participant MAE equally averages each participant's MAE; participant
RMSE equally averages their within-person RMSE. Paired differences use 2,000 participant-cluster
bootstrap draws with all fixed OOF predictions per selected participant. Negative wake-minus-fixed
MAE favors wake. Intervals condition on this exploratory workflow and fixed predictions; they do not
cover refitting/model-selection uncertainty or correct for multiple comparisons.

In [ ]:
metrics = pd.read_json(REPORT / "metrics.json")
intervals = pd.read_json(REPORT / "paired_intervals.json")
primary = metrics.loc[metrics.scenario.eq(nominal) & metrics.cohort.eq("pair_fixed_60") & metrics.fold.eq("pooled")]
display(primary[["model", "responses", "participants", "mae", "rmse", "participant_mae", "participant_rmse"]])
direct = intervals.loc[intervals.model.eq("wake") & intervals.baseline.str.startswith("fixed")]
display(direct[["scenario", "cohort", "baseline", "responses", "mae_difference", "mae_difference_low", "mae_difference_high", "participant_mae_difference"]])
display(intervals.loc[intervals.scenario.eq(nominal) & intervals.cohort.eq("pair_fixed_60") & intervals.model.eq("wake")])
display(Image(filename=str(REPORT / "performance.png")))

## Coverage helps interpret the weak results

Among the 227 primary included responses, the median wake interval is 115 minutes, yet median
accepted-reading occupancy is 48.1%, versus 58.3% for the 60-minute interval. Median largest gaps
including edges are 39.4 and 20.0 minutes. A latest reading about 0.93 seconds old in both does not
make the earlier recording complete. Partial minute bins use ceil(duration/60); bins are a reading-density
diagnostic, not continuously supported physiological time. No quality column enters the regressions.

In [ ]:
quality_table = pd.read_json(REPORT / "quality_diagnostics.json")
q = quality_table.loc[quality_table.scenario.eq(nominal) & quality_table.cohort.eq("pair_fixed_60") & quality_table.group.eq("included")]
display(q[["interval", "responses", "duration_seconds_median", "valid_readings_median", "occupied_bin_fraction_median",
           "observed_span_seconds_median", "latest_age_seconds_median", "left_gap_seconds_median",
           "maximum_internal_gap_seconds_median", "maximum_gap_including_edges_seconds_median"]])
display(Image(filename=str(REPORT / "recording_support.png")))

## Conclusion and next justified work

Nominal primary OOF MAE is **2.2506 for wake versus 2.2510 for 60 minutes**. Their difference is
−0.0003 points (conditional 95% interval −0.0275 to +0.0263). Matching median and context baselines
have lower point-estimate MAE, 2.1410 and 2.1858. Secondary comparisons, common cohort and later-minute
sensitivity also do not establish a wake advantage. This does not prove equivalence or absence of
physiological signal; it gives no reason to choose an optimal window or deploy this baseline.

Prioritize provider clarification of acquisition/receipt and batching, survey-start events, wake
report date/offset semantics and device-quality metadata. A stronger measurement/coverage protocol
needs a separate specification and evaluation. No messages were sent and no further experiment runs here.

Read the [report](../reports/morning_recall/b1fd2e61d20cdbe4/report.md), [professor summary](../reports/morning_recall/b1fd2e61d20cdbe4/professor_summary.md),
[fold-level errors](../reports/morning_recall/b1fd2e61d20cdbe4/fold_metrics.md), and
[verification](../reports/morning_recall/verification_4c844f737b74ee62.md).

Reproduce with `PYTHONPATH=src uv run python -m garmin_morning_recall.pipeline` after setting
`GARMIN_REFERENCE_PROJECT`. The guarded entrypoint verifies/reuses immutable data. Generate reports
with `garmin_morning_recall.report.render(ROOT, BUNDLE)`; independent validation is
`garmin_morning_recall.verify.verify(ROOT, BUNDLE, reference_path)`. This notebook is the aggregate
walkthrough. Its source stays output-free; executed copies belong in ignored `outputs/`.

In [ ]:
for name in ["accounting.parquet", "features.parquet", "quality.parquet", "membership.parquet", "out_of_fold_predictions.parquet", "training_audit.json"]:
    print("Ignored output:", BUNDLE.relative_to(ROOT) / name)
print("Report:", REPORT.relative_to(ROOT) / "report.md")
print("Professor summary:", REPORT.relative_to(ROOT) / "professor_summary.md")
print("All 32 final-test participants remain sealed. No commit or push.")